# Polieus | 03 · Regressão do tempo de finalização administrativa

O Polieus é uma proposta de tecnologia cidadã da equipe Politech para aproximar moradores e prefeituras. Uma calçada danificada pode impedir o deslocamento de quem usa cadeira de rodas. Na plataforma proposta, o morador registra a ocorrência, informa o local e acompanha o encaminhamento. A equipe municipal consulta as solicitações para compreender as demandas e explicar suas decisões à população.

Na disciplina de Artificial Intelligence & Chatbot, o grupo utiliza registros públicos do SP156, de São Paulo, para investigar duas questões. A primeira trata da composição dos pedidos em cada distrito e mês. A segunda trata do tempo entre a abertura e a finalização administrativa de uma solicitação. As duas análises partem dos mesmos oito serviços de acessibilidade e pavimentação, observados em outubro–dezembro de 2025 e julho–setembro de 2026. Os meses intermediários ficam fora do recorte.

**Pergunta deste notebook.** Ao abrir um pedido de um dos oito serviços, que duração administrativa o histórico permite estimar? A unidade é uma solicitação individual com finalização observada. O resultado pode orientar a leitura do histórico pelo cidadão e pelo gestor. Uma data prometida para o atendimento ou para a execução de uma obra exigiria outras evidências.

**Origem dos dados.** O notebook 01 seleciona os serviços e produz as tabelas de estudo. A publicação municipal permite consultar as edições de origem. [Solicitações do SP156 no Portal de Dados Abertos de São Paulo](https://dados.prefeitura.sp.gov.br/dataset/dados-do-sp156)

**Cobertura territorial e temática.** A plataforma recebe relatos de municípios paulistas. Este estudo utiliza os distritos da cidade de São Paulo. Distrito é uma divisão da capital. Aos seis serviços iniciais de acessibilidade e pavimentação somam-se limpeza de bueiros e manutenção de pontos de ônibus, que permitem estudar drenagem da via e condições de circulação. A cobertura representa os pedidos ao SP156 nos períodos selecionados.

### Roteiro de envio manual no Google Colab

| Etapa | Envie pelo painel Arquivos desta sessão | O que será produzido |
|---|---|---|
| 01 · Preparar os dados | `arquivo-final-4-trim-25.csv` e `arquivo-final-3-trim-26.csv` | Dois CSVs para modelagem e uma tabela de cobertura |
| 02 · Agrupar perfis | Somente `02_perfis_por_distrito_e_mes.csv` | Modelo de agrupamento e ficha técnica |
| 03 · Estimar dias | Somente `03_solicitacoes_com_duracao_em_dias.csv` | Modelo de regressão e ficha técnica |

Para reproduzir o estudo desde a origem, abra o notebook 01 pelo menu **Arquivo → Fazer upload de notebook** do Colab. Baixe as duas edições pelos links da primeira seção do 01, mantenha os nomes originais e arraste os CSVs para a pasta principal do painel **Arquivos**. Aguarde o fim do envio e clique em **Executar tudo**. Depois, abra `dados_polieus` e use **⋮ → Fazer download** para guardar as tabelas preparadas no computador.

Abra o 02 ou o 03 em sua própria sessão e arraste apenas o CSV indicado na linha correspondente. O ZIP já inclui esses dois CSVs. Quem quiser reproduzir diretamente um modelo pode usar a cópia incluída. A tabela `01_cobertura_por_servico_e_trimestre.csv` documenta o número de pedidos e a disponibilidade de duração por serviço. Guarde-a para interpretar os resultados. Em uma sessão nova, repita o envio do arquivo correspondente àquela etapa.

### Por que há três notebooks e dois modelos?

A prefeitura publica os chamados em arquivos trimestrais. Para estudar os perfis, precisamos de uma linha por distrito e mês. Para estudar os dias de atendimento, precisamos de uma linha por solicitação com duração conhecida. O notebook 01 transforma as mesmas fontes nessas duas tabelas, documenta as exclusões e permite conferir os totais antes de treinar. Dividir o estudo em três arquivos é uma escolha de organização da equipe. Os modelos aprendem nos notebooks 02 e 03.

O gestor quer saber quais territórios apresentam composições de pedidos semelhantes. Como o histórico não fornece grupos previamente identificados, usamos agrupamento, uma forma de aprendizagem não supervisionada. A segunda pergunta pede um número de dias. Como temos exemplos passados com abertura e finalização, usamos regressão, uma forma de aprendizagem supervisionada. Uma classificação exigiria definir uma categoria-alvo, como uma prioridade validada, e responderia a outra pergunta.

## 1. Primeira execução no Google Colab

1. Baixe o ZIP da entrega. No computador, extraia o ZIP e localize **`dados_polieus/03_solicitacoes_com_duracao_em_dias.csv`**. Se você executou o notebook 01, também pode obter essa tabela pelo painel **Arquivos → dados_polieus → ⋮ → Fazer download**.
2. Abra o [Google Colab](https://colab.research.google.com/) e entre com sua conta Google. Na janela de abertura, escolha **Upload** e selecione o arquivo `.ipynb`. Se já estiver em outro notebook, use **Arquivo → Fazer upload de notebook**. Para guardar suas alterações, salve uma cópia no Drive.
3. Clique em **Conectar**, se solicitado. Use o ambiente padrão Python com CPU. Clique no ícone de **pasta** à esquerda. Arraste **`03_solicitacoes_com_duracao_em_dias.csv`** do computador para o painel de arquivos, ao lado de `sample_data`. Aguarde até aparecer o nome completo. A célula de leitura também aceita a subpasta `dados_polieus`.
4. No menu **Ambiente de execução**, selecione **Executar tudo**. Cada caixa de código é uma célula. O resultado aparece logo abaixo dela. Se preferir estudar por partes, use o triângulo à esquerda de cada célula, de cima para baixo. Se surgir uma mensagem sobre o CSV, confira o nome e o local do upload e volte a executar a partir da leitura.
5. Ao encontrar a mensagem final de exportação verificada, abra a pasta **`modelos_polieus`** no painel lateral. No menu **⋮** de cada arquivo, escolha **Fazer download** para guardar o modelo e a ficha técnica antes de encerrar a sessão. Se o painel ainda não mostrar os arquivos, use o botão de atualização.

A entrada deste notebook é a tabela já preparada. Os CSVs municipais pesados são utilizados somente no notebook 01. Em uma nova sessão do Colab, envie novamente a tabela correspondente.



## 2. Leitura e significado das colunas

A tabela contém serviço, distrito, período de origem, data de abertura, data de finalização e duração em dias corridos. O notebook 01 selecionou registros com status **Finalizada** e intervalo observável maior ou igual a zero. Um pedido aberto e finalizado na mesma data tem duração zero. Verificamos o cálculo novamente antes de ajustar qualquer modelo.

Um pedido em aberto ainda pode vir a ter uma duração longa. Ao aprender apenas com desfechos conhecidos, o modelo pode subestimar a espera de parte da população. Para examinar a cobertura de todos os pedidos, consulte a tabela `01_cobertura_por_servico_e_trimestre.csv` do notebook 01. Ela permite recuperar o denominador de cada serviço e período.

### Os oito serviços utilizados

A seleção usa o nome registrado na fonte. Os exemplos ilustram situações de uso. Cada chamado entra uma vez na contagem de seu serviço.

1. **Tapa-buraco.** Pedido de reparo localizado no pavimento de uma via. Um exemplo é um buraco na pista de tráfego comum. O SP156 distingue os pedidos associados a corredores e faixas exclusivas de ônibus.

2. **Tapa-buraco em faixa exclusiva ou corredor de ônibus.** Pedido de reparo no pavimento de uma faixa exclusiva ou corredor de ônibus. A localização operacional distingue essa classe do tapa-buraco em via comum.

3. **Acessibilidade - solicitar avaliação de obstáculo na calçada.** Solicitação de avaliação de uma barreira no passeio que dificulta a circulação. O registro representa um pedido de vistoria ou avaliação. A remoção efetiva do obstáculo exige confirmação posterior.

4. **Calçada pública - solicitar manutenção.** Pedido de manutenção de uma calçada identificada como pública. Um exemplo é um trecho de passeio público com piso danificado que dificulta a passagem a pé.

5. **Calçada particular - denunciar calçada danificada ou inexistente.** Denúncia de passeio particular danificado ou inexistente. O trâmite pode envolver fiscalização. A duração administrativa precisa ser interpretada conforme esse tipo de atendimento.

6. **Guias para travessia de pedestres - Solicitar rebaixamento.** Pedido de rebaixamento de guia em travessia de pedestres, por exemplo para facilitar a passagem de cadeira de rodas entre a calçada e a pista.

7. **Solicitar limpeza de bueiros, boca de lobo e poços de visita.** Solicitação de limpeza de estruturas da drenagem urbana, como uma boca de lobo obstruída. A categoria permite observar demandas de escoamento de água em vias de circulação.

8. **Ponto de ônibus – Solicitar manutenção, conserto ou limpeza de equipamento existente.** Pedido de manutenção, conserto ou limpeza de equipamento já existente em ponto de ônibus. Um exemplo é um abrigo danificado. Reclamações sobre intervalos de linhas e pedidos de implantação têm outras categorias.

### Como ler os conceitos da regressão

**Regressão da duração administrativa** é a estimativa do número de dias corridos entre a abertura e a finalização administrativa de um pedido. Um pedido aberto no dia 2 e finalizado no dia 10 do mesmo mês fornece um exemplo observado de oito dias. Esse número é o **alvo**. Serviço, distrito quando conhecido e dia da semana da abertura são os **atributos de entrada**. Com eles, o modelo aprende a estimar o alvo de outros chamados. A finalização registrada permite medir o tempo administrativo. A confirmação de um reparo físico depende de vistoria ou registro da execução.

**Treino, validação e teste** têm papéis distintos. O treino ajusta o modelo com exemplos antigos. A validação orienta a escolha entre algoritmos. O teste mede o erro em um período posterior reservado. Usar a data de finalização como entrada revelaria uma informação futura, problema conhecido como **vazamento de dados**.

**Referência** é uma previsão simples para comparar resultados. Aqui ela usa a mediana histórica do serviço. **MAE** resume o tamanho médio das diferenças absolutas em dias. **RMSE** também é expresso em dias e dá maior peso aos erros grandes. **R²** compara o erro quadrático com o de uma previsão constante igual à média do conjunto avaliado. Um valor negativo indica erro quadrático superior ao dessa constante. O MAE de 4,31 dias no teste resume muitos chamados e não corresponde a uma margem garantida para cada morador.

**Pipeline** reúne as transformações das entradas e o estimador treinado. **Serialização** grava esse objeto em um arquivo .joblib. A API Flask reabre o arquivo e aplica as mesmas transformações a uma nova entrada, sem repetir o treinamento.

In [ ]:
from pathlib import Path
import json
import platform
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

In [ ]:
arquivo = Path("03_solicitacoes_com_duracao_em_dias.csv")
if not arquivo.is_file():
    arquivo = Path("dados_polieus") / arquivo
if not arquivo.is_file():
    raise FileNotFoundError("Envie 03_solicitacoes_com_duracao_em_dias.csv pelo painel Arquivos do Colab e execute novamente.")
dados = pd.read_csv(arquivo)
print("Linhas e colunas:", dados.shape)
display(dados.head(3))

campos = ["periodo_fonte", "servico", "distrito", "abertura", "finalizacao", "duracao_dias"]
if not set(campos).issubset(dados):
    raise ValueError("Confira se enviou 03_solicitacoes_com_duracao_em_dias.csv produzido pelo notebook 01.")
dados["abertura"] = pd.to_datetime(dados["abertura"], format="%Y-%m-%d", errors="raise")
dados["finalizacao"] = pd.to_datetime(dados["finalizacao"], format="%Y-%m-%d", errors="raise")
intervalos = (dados["finalizacao"] - dados["abertura"]).dt.days
if not ((dados.duracao_dias == intervalos) & (intervalos >= 0)).all():
    raise ValueError("Há datas ausentes ou uma duração incompatível com as datas.")
if dados.servico.isna().any():
    raise ValueError("O recorte exige um dos oito serviços identificados no notebook 01.")
print("Ausências por campo:")
display(dados[campos].isna().sum())
print("Linhas com a mesma combinação pública de campos (além da primeira):", dados[campos].duplicated().sum())

## 3. Simulação de uso na abertura de um pedido

Um sistema em operação só pode treinar com durações cujo desfecho já era conhecido. Usar a duração de um pedido ainda em aberto na data de corte revelaria o futuro ao algoritmo. Por isso controlamos **duas datas**, a de abertura e a de finalização.

| Etapa | Aberturas | Quais desfechos entram |
|---|---|---|
| Treinamento inicial | Outubro a dezembro de 2025 | Finalizações anteriores a 1º de julho de 2026 |
| Validação e escolha | Julho e agosto de 2026 | Finalizações anteriores a 1º de setembro de 2026 |
| Ajuste para o teste | Todas as aberturas do recorte anteriores a setembro de 2026 | Finalizações conhecidas antes de 1º de setembro |
| Teste final | Setembro de 2026 | Finalizações observadas na edição disponível |

O teste representa um período posterior na mesma cidade. Seu resultado depende de quanto tempo as solicitações puderam ser acompanhadas. A data de encerramento verifica o alvo e sua disponibilidade. As entradas efetivas são **serviço, distrito e dia da semana da abertura**. O distrito pressupõe que o território já possa ser conhecido a partir do local informado. Uma ausência fica identificada como categoria própria. O dia da semana é representado como categoria, com segunda-feira igual a 0 e domingo igual a 6.

In [ ]:
dados["dia_semana"] = dados["abertura"].dt.dayofweek.astype(str)
atributos = ["servico", "distrito", "dia_semana"]
alvo = "duracao_dias"
inicio_val = pd.Timestamp("2026-07-01")
inicio_teste = pd.Timestamp("2026-09-01")
t0 = dados.abertura.between("2025-10-01", "2025-12-31")
v0 = (dados.abertura >= inicio_val) & (dados.abertura < inicio_teste)
treino = dados.loc[t0 & (dados.finalizacao < inicio_val)].copy()
validacao = dados.loc[v0 & (dados.finalizacao < inicio_teste)].copy()
ajuste = dados.loc[(t0 | v0) & (dados.finalizacao < inicio_teste)].copy()
teste = dados.loc[dados.abertura.between(inicio_teste, "2026-09-30")].copy()
if min(len(treino), len(validacao), len(ajuste), len(teste)) == 0:
    raise ValueError("Confira se o CSV contém os dois trimestres selecionados.")
assert treino.finalizacao.max() < validacao.abertura.min()
assert ajuste.finalizacao.max() < teste.abertura.min()
print("Linhas de treino, validação, ajuste e teste:", len(treino), len(validacao), len(ajuste), len(teste))
print("Desfechos da tabela ainda indisponíveis em cada corte:", t0.sum() - len(treino), v0.sum() - len(validacao))

## 4. Exploração dos dados de treino

Começamos pelo período disponível para treinamento. Compare a mediana, que separa as duas metades da amostra, ao percentil 90, abaixo do qual ficam aproximadamente nove em cada dez durações. O máximo ajuda a identificar uma cauda longa. Uma duração alta e coerente com as datas pode representar um atendimento real. Os valores extremos permanecem na amostra.

Linhas com os mesmos campos públicos também permanecem. A fonte pode conter dois pedidos legítimos no mesmo distrito, para o mesmo serviço, abertos e finalizados nas mesmas datas. A ausência de um identificador individual impede confirmar que se trata do mesmo protocolo.

In [ ]:
display(treino.groupby("servico")[alvo].agg(
    quantidade="size", mediana="median", media="mean", p90=lambda s: s.quantile(.90), maximo="max"
).round(1))
print("Durações de zero dia no treino:", (treino[alvo] == 0).sum())
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(treino[alvo], bins=40, color="#216270")
ax.set_yscale("log")
ax.set(xlabel="Dias até a finalização", ylabel="Solicitações (escala logarítmica)", title="Distribuição das durações observadas no treino")
plt.tight_layout()
plt.show()

## 5. Referência e algoritmos candidatos

A **referência histórica** devolve a mediana de duração do serviço calculada apenas com dados de ajuste. Uma categoria sem histórico nessa referência receberia a mediana geral. Isso permite perguntar quanto se ganha ao usar informações adicionais.

Comparamos dois regressores. A **árvore de decisão** divide a amostra em grupos que apresentam durações diferentes. Limitamos a profundidade e exigimos um mínimo de exemplos por folha para reduzir a influência de combinações raras. O **gradient boosting** aprende uma sequência de árvores pequenas que corrigem os erros anteriores. Usamos perda absoluta nesse candidato, coerente com o objetivo de reduzir o erro típico em dias.

Cada candidato é um *pipeline*, ou seja, a preparação dos campos e o algoritmo ficam no mesmo objeto. Para a árvore, cada categoria recebe um indicador binário. Para o boosting, os códigos representam categorias identificadas explicitamente como tais. Categorias novas têm tratamento definido e as transformações aprendem apenas com os dados de ajuste. A saída numérica utiliza a mesma unidade do alvo.

In [ ]:
def preparar_categorias(codificador):
    return Pipeline([
        ("ausencias", SimpleImputer(strategy="constant", fill_value="SEM DISTRITO")),
        ("categorias", codificador),
    ])

arvore = Pipeline([
    ("preparo", preparar_categorias(OneHotEncoder(handle_unknown="ignore"))),
    ("regressor", DecisionTreeRegressor(max_depth=7, min_samples_leaf=50, random_state=42)),
])
boosting = Pipeline([
    ("preparo", preparar_categorias(OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan))),
    ("regressor", HistGradientBoostingRegressor(loss="absolute_error", categorical_features=[0, 1, 2],
        max_iter=100, max_leaf_nodes=15, min_samples_leaf=40, early_stopping=False, random_state=42)),
])
candidatos = {"Árvore": arvore, "Boosting": boosting}

def prever_mediana(historico, novos):
    medianas = historico.groupby("servico")[alvo].median()
    return novos.servico.map(medianas).fillna(historico[alvo].median()).to_numpy()

def medir(observado, previsto):
    return {"MAE (dias)": mean_absolute_error(observado, previsto),
            "RMSE (dias)": np.sqrt(mean_squared_error(observado, previsto)),
            "R²": r2_score(observado, previsto)}

## 6. Seleção pela validação

O **MAE** é a média dos erros absolutos. Um MAE de cinco dias significaria que a diferença absoluta entre estimativa e duração observada foi, em média, de cinco dias. Esse número é um exemplo de interpretação. O **RMSE** dá mais peso aos erros grandes. O **R²** compara o erro quadrático ao de uma constante igual à média da amostra avaliada e pode ser negativo.

Escolhemos entre os dois regressores pelo menor MAE de validação, com o RMSE como desempate. A mediana por serviço é o critério de utilidade prática. Quando um modelo perde para essa referência, a perda deve aparecer na conclusão, mesmo que seja o melhor entre os algoritmos treinados.

In [ ]:
linhas = {"Mediana por serviço": medir(validacao[alvo], prever_mediana(treino, validacao))}
for nome, candidato in candidatos.items():
    treinamento = clone(candidato).fit(treino[atributos], treino[alvo])
    linhas[nome] = medir(validacao[alvo], treinamento.predict(validacao[atributos]))
placar = pd.DataFrame(linhas).T
display(placar.round(3))
vencedor = placar.loc[list(candidatos)].sort_values(["MAE (dias)", "RMSE (dias)"]).index[0]
print("Regressor selecionado pela validação:", vencedor)

## 7. Teste em setembro e análise dos erros

Reajustamos a configuração escolhida com todos os desfechos conhecidos antes de setembro. A referência histórica recebe a mesma atualização. Usamos os dois métodos nas mesmas solicitações de setembro, mantidas fora da escolha do algoritmo.

Na tabela por serviço, examine o número de exemplos junto ao MAE. Uma categoria rara contribui pouco para a média geral. O **viés médio** é a previsão menos a duração observada. Um valor positivo indica superestimativa média. O gráfico usa a diagonal como referência de previsão exata. Para evitar sobreposição visual, sorteamos até 800 exemplos apenas para o desenho. As métricas usam todas as linhas de teste.

In [ ]:
modelo = clone(candidatos[vencedor]).fit(ajuste[atributos], ajuste[alvo])
previsoes = modelo.predict(teste[atributos])
referencia = prever_mediana(ajuste, teste)
resultados = pd.DataFrame({vencedor: medir(teste[alvo], previsoes),
                           "Mediana por serviço": medir(teste[alvo], referencia)}).T
display(resultados.round(3))
erros = teste[["servico", "distrito", alvo]].copy()
erros["estimativa"] = previsoes
erros["erro"] = previsoes - erros[alvo]
erros["absoluto"] = erros.erro.abs()
erros["absoluto_ref"] = np.abs(referencia - erros[alvo])
por_servico = erros.groupby("servico").agg(
    pedidos=("erro", "size"), MAE=("absoluto", "mean"), MAE_referencia=("absoluto_ref", "mean"),
    vies_em_dias=("erro", "mean"), p90_erro_absoluto=("absoluto", lambda s: s.quantile(.9)))
display(por_servico.round(2))
ganho = resultados.loc["Mediana por serviço", "MAE (dias)"] - resultados.loc[vencedor, "MAE (dias)"]
print(f"Redução do MAE frente à referência: {ganho:.2f} dias (valor negativo indica piora).")
amostra = erros.sample(n=min(800, len(erros)), random_state=42)
limite = max(amostra[alvo].max(), amostra.estimativa.max())
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(amostra[alvo], amostra.estimativa, alpha=.18, color="#216270")
ax.plot([0, limite], [0, limite], linestyle="--", color="black")
ax.set(xlabel="Duração observada (dias)", ylabel="Duração estimada (dias)", title="Solicitações com finalização observada em teste")
plt.tight_layout()
plt.show()

## 8. Alcance da estimativa para cidadãos e gestores

A duração observada termina na finalização administrativa. Para um morador, uma estimativa de sete dias, por exemplo, seria uma referência histórica sujeita a erro. A confirmação de que a calçada voltou a ser transitável depende de informação sobre a execução do serviço.

O notebook 01 registrou, no terceiro trimestre de 2026, duração elegível em 118 dos 1.192 pedidos de avaliação de obstáculo na calçada (9,9%). A leitura do erro dessa categoria precisa considerar os muitos pedidos sem desfecho elegível. Pedidos mais recentes tiveram menos oportunidade de apresentar uma espera longa no arquivo. A queda do tempo observado entre períodos pode, portanto, refletir o acompanhamento disponível. A tabela a seguir destaca quanto a previsão se afasta dos casos de cauda longa usando um limite definido **no ajuste**. Ela serve ao diagnóstico posterior ao teste.

In [ ]:
limiar = ajuste[alvo].quantile(.90)
faixas = np.where(erros[alvo] > limiar, "Acima do p90 histórico", "Até o p90 histórico")
display(erros.groupby(faixas).agg(pedidos=("erro","size"), MAE=("absoluto","mean"), vies=("erro","mean")).round(2))
print("Percentil 90 do ajuste:", limiar, "dias")
display(erros.nlargest(3, "absoluto")[["servico", alvo, "estimativa", "erro"]].round(1))

### Interpretação para a gestão

Observe o ganho em dias em relação à mediana por serviço, os erros por categoria e os pedidos mais demorados. Um resultado agregado pode esconder erros maiores em serviços de acessibilidade menos frequentes. O corte temporal simula a disponibilidade dos desfechos segundo as datas publicadas.

No teste de setembro (9.825 casos), o boosting teve MAE de 4,31 dias frente a 4,56 da referência. O R² negativo e a subestimativa média de 15,18 dias nos casos acima do percentil 90 histórico limitam o uso como orientação individual.

**Leitura da execução de referência.** Em setembro, o modelo apresentou erro absoluto médio de 4,31 dias, frente a 4,56 dias da mediana por serviço. O ganho médio foi de aproximadamente 0,25 dia. O R² de −0,114 indica que, pelo critério de erro quadrático, prever a média observada no próprio teste teria produzido erro menor. Essa média só fica conhecida depois do período. Uma estimativa individual deve ser apresentada como aproximação.

## 9. Exportação e uso da estimativa na API Flask

Guardamos o pipeline efetivamente avaliado. A API deverá receber o serviço, o distrito quando conhecido e a data de abertura. A partir dessa data, calcula o dia da semana com a mesma convenção abaixo e organiza as três entradas na ordem registrada na ficha. Um serviço fora do recorte deve ser informado como tal ao usuário. A API apresenta a estimativa em dias e explica sua finalidade administrativa.

O exemplo utiliza uma solicitação real de teste. Convertemos a data de abertura, verificamos a convenção do dia da semana e comparamos as respostas do modelo em memória e do arquivo recarregado. A ficha guarda as versões de software para que a API possa utilizar o mesmo ambiente que produziu o artefato. Para testar com suas próprias versões no Colab, gere e use o par de arquivos desta execução.

In [ ]:
pedido = teste.iloc[0]
entrada_api = pd.DataFrame([{
    "servico": pedido.servico, "distrito": pedido.distrito,
    "dia_semana": str(pd.Timestamp(pedido.abertura).dayofweek),
}], columns=atributos)
assert entrada_api.iloc[0]["dia_semana"] == pedido.dia_semana

In [ ]:
pasta = Path("modelos_polieus")
pasta.mkdir(exist_ok=True)
arquivo_modelo = pasta / "modelo_duracao_administrativa.joblib"
joblib.dump(modelo, arquivo_modelo, compress=3)
recuperado = joblib.load(arquivo_modelo)
exemplo = entrada_api
antes = modelo.predict(exemplo)
depois = recuperado.predict(exemplo)
np.testing.assert_allclose(antes, depois)
ficha = {
    "modelo": "polieus_regressao",
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "joblib": joblib.__version__,
    "entradas": atributos, "servicos": sorted(ajuste.servico.unique().tolist()),
    "dia_semana": "texto de 0 (segunda-feira) a 6 (domingo), calculado da abertura",
    "saida": "dias corridos até a finalização administrativa observada",
    "regressor": vencedor, "finalizacoes_de_ajuste_anteriores_a": "2026-09-01",
    "periodo_teste": "2026-09", "metricas_teste": resultados.to_dict("index"),
}
(pasta / "ficha_duracao_administrativa.json").write_text(json.dumps(ficha, ensure_ascii=False, indent=2), encoding="utf-8")
display(exemplo)
print("Estimativa do exemplo real em dias:", depois[0])
print("Exportação verificada.", arquivo_modelo)